# DopplerNet Self-Attention Benchmark
Two models: 1D Transformer + CQT ridge | 2D CNN + Transformer + CQT spectrogram

Tasks: vehicle trajectory path classification,
       source speed estimation (m/s),
       source distance estimation (m)

In [11]:
# ── Top-level control flags ───────────────────────────────────────────────────
TRAIN_ON    = "v2"     # "v1" → neurips_v1, MAX_DIST_M=100  |  "v2" → neurips_v2, MAX_DIST_M=1000

TRAIN_1D    = True     # Set False to skip 1D Transformer training
EVALUATE_1D = True     # Set False to skip 1D Transformer evaluation
TRAIN_2D    = True     # Set False to skip 2D CNN+Transformer training
EVALUATE_2D = True     # Set False to skip 2D CNN+Transformer evaluation

EPOCHS      = 300

### SECTION 1 — SETUP

In [12]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# !pip install librosa torch torchaudio tqdm --quiet

In [14]:
import os
import re
import glob
import random
import itertools
import math

import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict, Counter

import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Dataset: {TRAIN_ON}")

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

# ── Audio / CQT constants ─────────────────────────────────────────────────────
SR              = 22050
DURATION_S      = 10
N_SAMPLES       = SR * DURATION_S
HOP_LENGTH      = 512
BINS_PER_OCTAVE = 24
N_BINS          = 84
FMIN            = librosa.note_to_hz("C2")
MAX_T           = 432

# ── Dataset-dependent paths and constants ─────────────────────────────────────
ROOT = "/content/drive/Shareddrives/Spectral Transformers - Doppler"

assert TRAIN_ON in ("v1", "v2"), "TRAIN_ON must be 'v1' or 'v2'"

if TRAIN_ON == "v1":
    DATA_ROOT    = os.path.join(ROOT, "Datasets", "neurips_v1", "audio_clips")
    MODEL_ROOT   = os.path.join(ROOT, "doppler_models",  "v1_attn_benchmark")
    RESULTS_DIR  = os.path.join(ROOT, "results",         "v1_attn_benchmark")
    MAX_DIST_M   = 120.0
    DIST_BINS    = [(0, 20), (20, 40), (40, 60), (60, 100), (100, 130)]
    BIN_LABELS   = ["0-20", "20-40", "40-60", "60-100", "100-130"]
else:
    DATA_ROOT    = os.path.join(ROOT, "Datasets", "neurips_v2", "audio_clips")
    MODEL_ROOT   = os.path.join(ROOT, "doppler_models",  "v2_attn_benchmark")
    RESULTS_DIR  = os.path.join(ROOT, "results",         "v2_attn_benchmark")
    MAX_DIST_M   = 1000.0
    DIST_BINS    = [(0, 100), (100, 250), (250, 500), (500, 750), (750, 1010)]
    BIN_LABELS   = ["0-100", "100-250", "250-500", "500-750", "750-1000"]

MAX_SPEED_MPS = 50.0

RESULTS_DIR_1D  = os.path.join(RESULTS_DIR, "1d_attn")
RESULTS_DIR_2D  = os.path.join(RESULTS_DIR, "2d_attn")
RESULTS_DIR_CMP = os.path.join(RESULTS_DIR, "1d_vs_2d")
FIG_DIR_1D      = os.path.join(RESULTS_DIR_1D, "figures")
FIG_DIR_2D      = os.path.join(RESULTS_DIR_2D, "figures")

for d in [
    os.path.join(MODEL_ROOT, "attn_model_1d"),
    os.path.join(MODEL_ROOT, "attn_model_2d"),
    FIG_DIR_1D,
    FIG_DIR_2D,
    os.path.join(RESULTS_DIR_CMP, "figures"),
]:
    os.makedirs(d, exist_ok=True)

CKPT_1D = os.path.join(MODEL_ROOT, "attn_model_1d", "attn_1d.pt")
CKPT_2D = os.path.join(MODEL_ROOT, "attn_model_2d", "attn_2d.pt")
CSV_1D  = os.path.join(RESULTS_DIR_1D, "attn_results_1d.csv")
CSV_2D  = os.path.join(RESULTS_DIR_2D, "attn_results_2d.csv")

CQT_FREQS = librosa.cqt_frequencies(
    n_bins=N_BINS, fmin=FMIN, bins_per_octave=BINS_PER_OCTAVE
)
NYQUIST = SR / 2.0

print(f"Checkpoints : {MODEL_ROOT}")
print(f"Results     : {RESULTS_DIR}")
print(f"MAX_DIST_M  : {MAX_DIST_M}")

# ── Training hyperparameters ──────────────────────────────────────────────────
TRAIN_CLIPS = 1000
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
BATCH_SIZE  = 16
LR          = 3e-4
SAVE_EVERY  = 20

# ── Label parser ──────────────────────────────────────────────────────────────
LABEL_RE = re.compile(
    r'(?P<vehicle>[^_]+)_(?P<path>[^_]+)_'
    r'(?P<speed>[\d\.]+)mps_(?P<dist>[\d\.]+)m_'
)
PATH_TO_ID = {"straight": 0, "parabola": 1, "bezier": 2}
ID_TO_PATH = {v: k for k, v in PATH_TO_ID.items()}


def parse_label(fname):
    m = LABEL_RE.search(fname)
    if m is None:
        raise ValueError(f"Unrecognised filename: {fname}")
    return {
        "path":    m.group("path"),
        "path_id": PATH_TO_ID[m.group("path")],
        "speed":   float(m.group("speed")),
        "dist":    float(m.group("dist")),
    }


def get_label_key(wav_path):
    basename = os.path.basename(wav_path)
    try:
        return parse_label(basename)
    except ValueError:
        parent = os.path.basename(os.path.dirname(wav_path))
        return parse_label(parent)


def build_splits(data_root, n_clips, train_r, val_r, seed=42):
    all_wav = sorted(
        glob.glob(f"{data_root}/**/*.wav", recursive=True)
      + glob.glob(f"{data_root}/**/*.WAV", recursive=True)
    )
    assert len(all_wav) > 0, f"No .wav files found under: {data_root}"

    parseable = []
    for f in all_wav:
        try:
            get_label_key(f)
            parseable.append(f)
        except ValueError:
            pass

    buckets = defaultdict(list)
    for f in parseable:
        lbl = get_label_key(f)["path"]
        buckets[lbl].append(f)

    rng = random.Random(seed)
    for cls_files in buckets.values():
        rng.shuffle(cls_files)

    per_class_budget = n_clips // len(buckets)
    per_class = min(per_class_budget, min(len(v) for v in buckets.values()))

    balanced = []
    for cls_files in buckets.values():
        balanced.extend(cls_files[:per_class])
    rng.shuffle(balanced)

    n       = len(balanced)
    n_train = int(train_r * n)
    n_val   = int(val_r   * n)

    return (
        balanced[:n_train],
        balanced[n_train : n_train + n_val],
        balanced[n_train + n_val :],
    )


train_files, val_files, test_files = build_splits(
    DATA_ROOT, TRAIN_CLIPS, TRAIN_RATIO, VAL_RATIO
)

print(f"Train : {len(train_files)} clips")
print(f"Val   : {len(val_files)} clips")
print(f"Test  : {len(test_files)} clips")
print("Path distribution (train):",
      dict(Counter(get_label_key(f)["path"] for f in train_files)))

# ── Feature extraction ────────────────────────────────────────────────────────

def _npy_path(wav_path, name):
    return os.path.join(os.path.dirname(wav_path), name)


def _load_or_none(wav_path, name):
    p = _npy_path(wav_path, name)
    return np.load(p) if os.path.exists(p) else None


def pad_or_trim_time(arr, max_t):
    t = arr.shape[-1]
    if t >= max_t:
        return arr[..., :max_t]
    pad_cfg = [(0, 0)] * (arr.ndim - 1) + [(0, max_t - t)]
    return np.pad(arr, pad_cfg)


def _compute_cqt_log1p(wav):
    C = librosa.cqt(wav, sr=SR, hop_length=HOP_LENGTH,
                    fmin=FMIN, n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE)
    return np.log1p(np.abs(C))


def _compute_dfdt(f_norm):
    dfdt     = np.zeros_like(f_norm)
    dt       = HOP_LENGTH / SR
    dfdt[1:] = (f_norm[1:] - f_norm[:-1]) / dt
    max_abs  = np.max(np.abs(dfdt)) + 1e-8
    return dfdt / max_abs


def _align(arr, target_len):
    if len(arr) >= target_len:
        return arr[:target_len]
    return np.pad(arr, (0, target_len - len(arr)))


def extract_1d_features(wav, wav_path=None, augment=False):
    """7-channel Doppler trajectory → (7, MAX_T)."""
    freq_norm = _load_or_none(wav_path, "frequency.npy") if wav_path else None
    if freq_norm is None:
        logC      = _compute_cqt_log1p(wav)
        bins      = np.argmax(logC, axis=0)
        f_hz      = CQT_FREQS[bins]
        f_hz      = scipy.signal.medfilt(np.nan_to_num(f_hz, nan=0.0), kernel_size=5)
        freq_norm = np.clip(f_hz / NYQUIST, 0.0, 1.0)
    else:
        freq_norm = freq_norm.flatten()
        freq_norm = scipy.signal.medfilt(np.nan_to_num(freq_norm, nan=0.0), kernel_size=5)
        freq_norm = np.clip(freq_norm, 0.0, 1.0)

    freq_norm = freq_norm.flatten()
    if augment:
        freq_norm = np.clip(freq_norm + np.random.normal(0, 0.004, size=freq_norm.shape), 0.0, 1.0)

    dfdt = _load_or_none(wav_path, "dfdt.npy") if wav_path else None
    dfdt = dfdt.flatten() if dfdt is not None else _compute_dfdt(freq_norm)

    rms = _load_or_none(wav_path, "rms.npy") if wav_path else None
    if rms is None:
        rms = librosa.feature.rms(y=wav, frame_length=HOP_LENGTH * 2, hop_length=HOP_LENGTH)[0]
        rms = rms / (np.max(rms) + 1e-8)
    rms = rms.flatten()

    topk = _load_or_none(wav_path, "spec_topk.npy") if wav_path else None
    topk_freq = topk[:, 0, 0] if (topk is not None and topk.ndim == 3
                                   and topk.shape[1] >= 1 and topk.shape[2] >= 1) else freq_norm.copy()

    T         = len(dfdt)
    freq_norm = _align(freq_norm, T)
    rms       = _align(rms,       T)
    topk_freq = _align(topk_freq, T)

    dfdt2     = np.gradient(dfdt) / (np.std(np.gradient(dfdt)) + 1e-8)
    sign_dfdt = np.sign(dfdt)
    t_rel     = np.linspace(-1.0, 1.0, T)

    feat = np.stack([dfdt, dfdt2, sign_dfdt, freq_norm, rms, topk_freq, t_rel], axis=0)
    feat = pad_or_trim_time(feat, MAX_T)

    if augment:
        feat = np.roll(feat, np.random.randint(-20, 21), axis=-1)

    return feat.astype(np.float32)


def extract_2d_features(wav, wav_path=None):
    """Log-CQT spectrogram z-scored per bin → (1, 84, MAX_T)."""
    logC = _load_or_none(wav_path, "cqt.npy") if wav_path else None
    logC = _compute_cqt_log1p(wav) if logC is None else logC.astype(np.float32)
    logC = np.nan_to_num(logC)
    mean = logC.mean(axis=1, keepdims=True)
    std  = logC.std(axis=1,  keepdims=True) + 1e-6
    logC = (logC - mean) / std
    logC = pad_or_trim_time(logC, MAX_T)
    return logC[np.newaxis].astype(np.float32)


# ── Dataset classes ───────────────────────────────────────────────────────────

class _DopplerBase(Dataset):
    def __init__(self, files, augment=False):
        self.files   = files
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def _extract(self, wav, wav_path, augment):
        raise NotImplementedError

    def __getitem__(self, idx):
        fpath      = self.files[idx]
        wav, _     = librosa.load(fpath, sr=SR, mono=True)
        if self.augment:
            wav = wav * np.random.uniform(0.85, 1.15)
        x   = self._extract(wav, fpath, self.augment)
        lbl = get_label_key(fpath)
        return (
            torch.from_numpy(x),
            torch.tensor(lbl["path_id"], dtype=torch.long),
            torch.tensor(lbl["speed"],   dtype=torch.float32),
            torch.tensor(lbl["dist"],    dtype=torch.float32),
            os.path.basename(fpath),
        )


class DopplerDataset1D(_DopplerBase):
    def _extract(self, wav, wav_path, augment):
        return extract_1d_features(wav, wav_path, augment=augment)


class DopplerDataset2D(_DopplerBase):
    def _extract(self, wav, wav_path, augment):
        return extract_2d_features(wav, wav_path)


# ── Shared training infrastructure ───────────────────────────────────────────

_ce    = nn.CrossEntropyLoss(label_smoothing=0.05)
_huber = nn.SmoothL1Loss(beta=0.5)


def compute_loss(p_hat, s_hat, d_hat, p_gt, s_gt, d_gt):
    loss_path  = _ce(p_hat, p_gt)
    loss_speed = _huber(s_hat, s_gt / MAX_SPEED_MPS)
    loss_dist  = _huber(d_hat, torch.log1p(d_gt) / np.log1p(MAX_DIST_M))
    return loss_path + 2.5 * loss_speed + 1.5 * loss_dist


def save_ckpt(path, model, optimizer, scheduler,
              epoch, batch_idx, running_loss, best_combined):
    torch.save({
        "model":         model.state_dict(),
        "optimizer":     optimizer.state_dict(),
        "scheduler":     scheduler.state_dict(),
        "epoch":         epoch,
        "batch_idx":     batch_idx,
        "running_loss":  running_loss,
        "best_combined": best_combined,
    }, path)


def load_ckpt(path, model, optimizer, scheduler):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    return ckpt["epoch"], ckpt["batch_idx"], ckpt["running_loss"], ckpt.get("best_combined", float("inf"))


def build_epoch_loader(dataset, batch_size, epoch_idx):
    g = torch.Generator()
    g.manual_seed(42 + epoch_idx * 997)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True,
                      generator=g, num_workers=0, pin_memory=(DEVICE == "cuda"))


def run_validation(model, val_loader):
    model.eval()
    correct, speed_errs, dist_errs = 0, [], []
    with torch.no_grad():
        for x, p_gt, s_gt, d_gt, _ in val_loader:
            x = x.to(DEVICE)
            p_hat, s_hat, d_hat = model(x)
            pred_path  = p_hat.argmax(1).item()
            pred_speed = s_hat.item() * MAX_SPEED_MPS
            pred_dist  = max(0.0, torch.expm1(d_hat * np.log1p(MAX_DIST_M)).item())
            correct   += int(pred_path == p_gt.item())
            speed_errs.append(abs(pred_speed - s_gt.item()))
            dist_errs.append( abs(pred_dist  - d_gt.item()))
    path_acc = 100.0 * correct / max(1, len(val_loader))
    s_mae    = float(np.mean(speed_errs))
    d_mae    = float(np.mean(dist_errs))
    combined = s_mae / MAX_SPEED_MPS + d_mae / MAX_DIST_M + (1.0 - path_acc / 100.0)
    return path_acc, s_mae, d_mae, combined


def train_model(model_name, model, DatasetCls, ckpt_path,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                save_every=SAVE_EVERY, val_every=20):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-6)

    start_epoch   = 0
    resume_batch  = 0
    resume_loss   = 0.0
    best_combined = float("inf")

    if os.path.exists(ckpt_path):
        start_epoch, resume_batch, resume_loss, best_combined = load_ckpt(
            ckpt_path, model, optimizer, scheduler)
        if start_epoch >= epochs:
            print(f"[{model_name}] Already trained for {epochs} epochs — skipping.")
            return model
        print(f"[{model_name}] Resumed from epoch {start_epoch + 1}, "
              f"batch {resume_batch}  "
              f"(loss acc = {resume_loss:.4f}, best combined = {best_combined:.4f})")
    else:
        print(f"[{model_name}] Starting fresh training")

    train_ds   = DatasetCls(train_files, augment=True)
    val_ds     = DatasetCls(val_files,   augment=False)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
    model.to(DEVICE)

    for epoch in range(start_epoch, epochs):
        loader    = build_epoch_loader(train_ds, batch_size, epoch)
        n_batches = len(loader)
        midpoint  = n_batches // 2
        running   = resume_loss  if epoch == start_epoch else 0.0
        skip      = resume_batch if epoch == start_epoch else 0
        mid_saved = skip > midpoint
        model.train()

        from tqdm.auto import tqdm as _tqdm
        _iter = _tqdm(itertools.islice(loader, skip, None),
                      total=n_batches - skip,
                      desc=f"[{model_name}] Ep {epoch+1:03d}/{epochs}",
                      unit="batch", leave=False)

        for i, (x, p, s, d, _) in enumerate(_iter, start=skip):
            x = x.to(DEVICE); p = p.to(DEVICE)
            s = s.to(DEVICE); d = d.to(DEVICE)
            optimizer.zero_grad()
            p_hat, s_hat, d_hat = model(x)
            loss = compute_loss(p_hat, s_hat, d_hat, p, s, d)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            running += loss.item()
            _iter.set_postfix(loss=f"{running/(i+1):.4f}",
                              lr=f"{optimizer.param_groups[0]['lr']:.1e}")

            if i == midpoint and not mid_saved:
                save_ckpt(ckpt_path, model, optimizer, scheduler,
                          epoch, i + 1, running, best_combined)
                mid_saved = True
                print(f"  [{model_name}] ep {epoch+1}  "
                      f"[mid save @ batch {i+1}/{n_batches}]  "
                      f"loss {running/(i+1):.4f}")
            elif (i + 1) % save_every == 0:
                save_ckpt(ckpt_path, model, optimizer, scheduler,
                          epoch, i + 1, running, best_combined)
                print(f"  [{model_name}] ep {epoch+1}  "
                      f"batch {i+1}/{n_batches}  "
                      f"loss {running/(i+1):.4f}  [ckpt]")

        avg_loss     = running / max(1, n_batches)
        current_lr   = optimizer.param_groups[0]["lr"]
        is_val_epoch = ((epoch + 1) % val_every == 0 or epoch == epochs - 1)

        if is_val_epoch:
            path_acc, s_mae, d_mae, combined = run_validation(model, val_loader)
            print(f"[{model_name}] Epoch {epoch+1:02d}/{epochs}  "
                  f"train_loss {avg_loss:.4f}  lr {current_lr:.2e}  "
                  f"── VAL ── path {path_acc:.1f}%  "
                  f"speed MAE {s_mae:.2f} m/s  dist MAE {d_mae:.2f} m  "
                  f"combined {combined:.4f}")
            if combined < best_combined:
                best_combined = combined
                save_ckpt(ckpt_path, model, optimizer, scheduler,
                          epoch + 1, 0, 0.0, best_combined)
                print(f"  ↳ best combined {combined:.4f} — checkpoint updated")
            else:
                save_ckpt(ckpt_path, model, optimizer, scheduler,
                          epoch + 1, 0, 0.0, best_combined)
        else:
            print(f"[{model_name}] Epoch {epoch+1:02d}/{epochs}  "
                  f"train_loss {avg_loss:.4f}  lr {current_lr:.2e}")
            save_ckpt(ckpt_path, model, optimizer, scheduler,
                      epoch + 1, 0, 0.0, best_combined)

        scheduler.step(epoch + 1)
        resume_batch = 0
        resume_loss  = 0.0

    print(f"[{model_name}] Training complete.  Best combined: {best_combined:.4f}")
    return model


def reload_model(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model


def run_inference(model_name, model, DatasetCls):
    test_ds = DatasetCls(test_files, augment=False)
    loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)
    rows    = []
    with torch.no_grad():
        for x, p_gt, s_gt, d_gt, fname in loader:
            x = x.to(DEVICE)
            p_hat, s_hat, d_hat = model(x)
            pred_path  = p_hat.argmax(1).item()
            pred_speed = s_hat.item() * MAX_SPEED_MPS
            pred_dist  = max(0.0, torch.expm1(d_hat * np.log1p(MAX_DIST_M)).item())
            s_err = pred_speed - s_gt.item()
            d_err = pred_dist  - d_gt.item()
            rows.append({
                "model":            model_name,
                "file":             fname[0],
                "path_gt":          ID_TO_PATH[p_gt.item()],
                "path_pred":        ID_TO_PATH[pred_path],
                "path_correct":     int(pred_path == p_gt.item()),
                "speed_gt":         round(s_gt.item(), 3),
                "speed_pred":       round(pred_speed, 3),
                "speed_err":        round(abs(s_err), 3),
                "speed_err_signed": round(s_err, 3),
                "dist_gt":          round(d_gt.item(), 3),
                "dist_pred":        round(pred_dist, 3),
                "dist_err":         round(abs(d_err), 3),
                "dist_err_signed":  round(d_err, 3),
            })
    return rows


def plot_results(df, model_names, colors, fig_dir, speed_lim, dist_lim,
                 dist_bins, bin_labels):
    """Shared plotting routine — same graphs as CNN benchmark plus R² panels."""

    path_names = ["straight", "parabola", "bezier"]

    # ── Confusion matrices ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, len(model_names), figsize=(6 * len(model_names), 5))
    if len(model_names) == 1:
        axes = [axes]
    fig.suptitle("Path Classification — Confusion Matrices", fontsize=13)
    for ax, mname in zip(axes, model_names):
        sub  = df[df["model"] == mname]
        conf = np.zeros((3, 3), dtype=int)
        for _, row in sub.iterrows():
            conf[PATH_TO_ID[row["path_gt"]], PATH_TO_ID[row["path_pred"]]] += 1
        acc        = 100 * np.trace(conf) / conf.sum()
        row_totals = conf.sum(axis=1)
        col_totals = conf.sum(axis=0)
        ax.imshow(conf, cmap="Blues", vmin=0)
        ax.set_xticks(range(3))
        ax.set_xticklabels([f"{c}\n(n={col_totals[i]})" for i, c in enumerate(path_names)],
                           rotation=20, fontsize=9)
        ax.set_yticks(range(3))
        ax.set_yticklabels([f"{c} (n={row_totals[i]})" for i, c in enumerate(path_names)],
                           fontsize=9)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"{mname}   acc = {acc:.1f}%")
        for r in range(3):
            for c in range(3):
                ax.text(c, r, str(conf[r, c]), ha="center", va="center", fontsize=12,
                        color="white" if conf[r, c] > conf.max() * 0.6 else "black")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "confusion_matrices.png"), dpi=150)
    plt.show()

    # ── Speed scatter ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, len(model_names), figsize=(6 * len(model_names), 4.5))
    if len(model_names) == 1:
        axes = [axes]
    fig.suptitle("Speed Estimation — GT vs Predicted", fontsize=13)
    for ax, mname in zip(axes, model_names):
        sub  = df[df["model"] == mname]
        ax.scatter(sub["speed_gt"], sub["speed_pred"], s=10, alpha=0.45, color=colors[mname])
        ax.plot(speed_lim, speed_lim, "k--", linewidth=1, label="y = x")
        ax.set_xlim(speed_lim); ax.set_ylim(speed_lim)
        ax.set_xlabel("GT speed (m/s)"); ax.set_ylabel("Pred speed (m/s)")
        mae  = sub["speed_err"].mean()
        bias = sub["speed_err_signed"].mean()
        ss_res = ((sub["speed_pred"] - sub["speed_gt"]) ** 2).sum()
        ss_tot = ((sub["speed_gt"]   - sub["speed_gt"].mean()) ** 2).sum()
        r2 = 1 - ss_res / (ss_tot + 1e-8)
        ax.set_title(f"{mname}   MAE={mae:.2f}  bias={bias:+.2f}  R²={r2:.3f}")
        ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "speed_scatter.png"), dpi=150)
    plt.show()

    # ── Distance scatter ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, len(model_names), figsize=(6 * len(model_names), 4.5))
    if len(model_names) == 1:
        axes = [axes]
    fig.suptitle("Distance Estimation — GT vs Predicted", fontsize=13)
    for ax, mname in zip(axes, model_names):
        sub  = df[df["model"] == mname]
        ax.scatter(sub["dist_gt"], sub["dist_pred"], s=10, alpha=0.45, color=colors[mname])
        ax.plot(dist_lim, dist_lim, "k--", linewidth=1, label="y = x")
        ax.set_xlim(dist_lim); ax.set_ylim(dist_lim)
        ax.set_xlabel("GT distance (m)"); ax.set_ylabel("Pred distance (m)")
        mae  = sub["dist_err"].mean()
        bias = sub["dist_err_signed"].mean()
        ss_res = ((sub["dist_pred"] - sub["dist_gt"]) ** 2).sum()
        ss_tot = ((sub["dist_gt"]   - sub["dist_gt"].mean()) ** 2).sum()
        r2 = 1 - ss_res / (ss_tot + 1e-8)
        ax.set_title(f"{mname}   MAE={mae:.2f}  bias={bias:+.2f}  R²={r2:.3f}")
        ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "distance_scatter.png"), dpi=150)
    plt.show()

    # ── MAE bar chart ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    fig.suptitle("Mean Absolute Error", fontsize=13)
    for ax, (col, unit) in zip(axes, [("speed_err", "Speed MAE (m/s)"),
                                       ("dist_err",  "Distance MAE (m)")]):
        vals = [df[df["model"] == m][col].mean() for m in model_names]
        bars = ax.bar(model_names, vals, color=[colors[m] for m in model_names],
                      edgecolor="white", linewidth=0.8)
        ax.set_ylabel(unit); ax.set_title(unit)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.02 * max(vals),
                    f"{val:.2f}", ha="center", va="bottom", fontsize=11)
        ax.set_ylim(0, max(vals) * 1.25)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "mae_comparison.png"), dpi=150)
    plt.show()

    # ── Distance MAE by range bin ─────────────────────────────────────────────
    fig, axes = plt.subplots(1, len(model_names), figsize=(6 * len(model_names), 4.5))
    if len(model_names) == 1:
        axes = [axes]
    fig.suptitle("Distance MAE by Range Bin", fontsize=13)
    for ax, mname in zip(axes, model_names):
        sub      = df[df["model"] == mname]
        maes, ns = [], []
        for lo, hi in dist_bins:
            mask = (sub["dist_gt"] >= lo) & (sub["dist_gt"] < hi)
            maes.append(sub[mask]["dist_err"].mean() if mask.sum() > 0 else 0.0)
            ns.append(int(mask.sum()))
        bars = ax.bar(bin_labels, maes, color=colors[mname])
        ax.set_xlabel("GT distance range (m)"); ax.set_ylabel("MAE (m)")
        ax.set_title(mname); ax.tick_params(axis="x", rotation=25)
        for bar, val, n in zip(bars, maes, ns):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.3,
                        f"{val:.1f}\n(n={n})", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "dist_mae_by_range.png"), dpi=150)
    plt.show()

    # ── Error histograms ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, len(model_names), figsize=(6 * len(model_names), 8))
    if len(model_names) == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle("Error Distribution Histograms (signed)", fontsize=13)
    for col_idx, mname in enumerate(model_names):
        sub = df[df["model"] == mname]
        for row_idx, (err_col, unit, xlim) in enumerate([
            ("speed_err_signed", "Speed error (m/s)", (-MAX_SPEED_MPS, MAX_SPEED_MPS)),
            ("dist_err_signed",  "Dist error (m)",    (-MAX_DIST_M,    MAX_DIST_M)),
        ]):
            ax   = axes[row_idx, col_idx]
            errs = sub[err_col].values
            b    = np.linspace(xlim[0], xlim[1], 35)
            ax.hist(errs, bins=b, color=colors[mname], alpha=0.75, edgecolor="white")
            ax.axvline(0,               color="black",  linewidth=1.5, linestyle="--", label="zero")
            ax.axvline(errs.mean(),     color="red",    linewidth=1.2, linestyle="-",
                       label=f"mean={errs.mean():+.2f}")
            ax.axvline(np.median(errs), color="orange", linewidth=1.2, linestyle="-",
                       label=f"med={np.median(errs):+.2f}")
            ax.set_xlabel(unit, fontsize=10); ax.set_ylabel("Count", fontsize=10)
            ax.set_title(f"{mname} — {unit}", fontsize=10)
            ax.legend(fontsize=8); ax.set_xlim(xlim)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "error_histograms.png"), dpi=150)
    plt.show()

    # ── Per-path MAE ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle("Per-Path MAE Breakdown", fontsize=13)
    x = np.arange(len(path_names))
    w = 0.35 / max(len(model_names), 1)
    for ax, (err_col, unit) in zip(axes, [("speed_err", "Speed MAE (m/s)"),
                                           ("dist_err",  "Distance MAE (m)")]):
        for i, mname in enumerate(model_names):
            sub    = df[df["model"] == mname]
            vals   = [sub[sub["path_gt"] == p][err_col].mean() for p in path_names]
            offset = (i - (len(model_names) - 1) / 2) * w
            bars   = ax.bar(x + offset, vals, width=w, label=mname,
                            color=colors[mname], alpha=0.85)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.01 * max(vals),
                        f"{val:.1f}", ha="center", va="bottom", fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(path_names)
        ax.set_ylabel(unit); ax.set_title(unit)
        ax.legend(fontsize=9)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.2)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "per_path_mae.png"), dpi=150)
    plt.show()

    # ── R² bar chart (extra metric vs CNN benchmark) ──────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    fig.suptitle("R² Score (higher = better)", fontsize=13)
    for ax, (val_col, gt_col, unit) in zip(axes, [
        ("speed_pred", "speed_gt", "Speed R²"),
        ("dist_pred",  "dist_gt",  "Distance R²"),
    ]):
        r2_vals = []
        for mname in model_names:
            sub    = df[df["model"] == mname]
            ss_res = ((sub[val_col] - sub[gt_col]) ** 2).sum()
            ss_tot = ((sub[gt_col]  - sub[gt_col].mean()) ** 2).sum()
            r2_vals.append(1 - ss_res / (ss_tot + 1e-8))
        bars = ax.bar(model_names, r2_vals, color=[colors[m] for m in model_names],
                      edgecolor="white", linewidth=0.8)
        ax.set_ylabel(unit); ax.set_title(unit)
        ax.set_ylim(min(0, min(r2_vals)) - 0.05, 1.05)
        ax.axhline(1.0, color="gray", linewidth=0.8, linestyle="--")
        for bar, val in zip(bars, r2_vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "r2_scores.png"), dpi=150)
    plt.show()


print("Section 1 complete — setup, features, and helpers loaded.")


MODEL_NAME_1D = f"1D-Attn-{TRAIN_ON}"
MODEL_NAME_2D = f"2D-Attn-{TRAIN_ON}"
COLORS        = {MODEL_NAME_1D: "steelblue", MODEL_NAME_2D: "coral"}
SPEED_LIM     = [0, MAX_SPEED_MPS + 5]
DIST_LIM      = [0, MAX_DIST_M + 10]

Device : cuda
Dataset: v2
Checkpoints : /content/drive/Shareddrives/Spectral Transformers - Doppler/doppler_models/v2_attn_benchmark
Results     : /content/drive/Shareddrives/Spectral Transformers - Doppler/results/v2_attn_benchmark
MAX_DIST_M  : 1000.0
Train : 799 clips
Val   : 99 clips
Test  : 101 clips
Path distribution (train): {'straight': 268, 'parabola': 273, 'bezier': 258}
Section 1 complete — setup, features, and helpers loaded.


In [15]:
class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])

def _task_head(in_dim, out_dim):
    return nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, out_dim))

## DopplerTransformer1D

### SECTION 2A — DopplerTransformer1D

In [ ]:
# Architecture:
#   Input  : (B, 7, 432)  — 7-channel Doppler trajectory
#   Project: Linear 7 → d_model=128 per time step   → (B, 432, 128)
#   + Sinusoidal positional encoding
#   Encoder: 3-layer Transformer encoder
#             (nhead=4, dim_feedforward=256, dropout=0.1, norm_first=True)
#   Pool   : mean over time  → (B, 128)
#   Shared : LayerNorm → Linear(128) → ReLU → Dropout(0.2)
#   Heads  : path (3), speed (1), dist (1)

class DopplerTransformer1D(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=3,
                 dim_ff=256, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(7, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=MAX_T, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.shared = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.path_head  = _task_head(d_model, 3)
        self.speed_head = _task_head(d_model, 1)
        self.dist_head  = _task_head(d_model, 1)

    def forward(self, x):
        # x: (B, 7, T)
        x = x.permute(0, 2, 1)           # (B, T, 7)
        x = self.input_proj(x)            # (B, T, d_model)
        x = self.pos_enc(x)               # (B, T, d_model)
        x = self.encoder(x)               # (B, T, d_model)
        g = x.mean(dim=1)                 # (B, d_model)  — mean pool over time
        z = self.shared(g)
        return (self.path_head(z),
                self.speed_head(z).squeeze(1),
                self.dist_head(z).squeeze(1))


if TRAIN_1D:
    model_1d = DopplerTransformer1D().to(DEVICE)
    model_1d = train_model(
        model_name = f"1D-Attn-{TRAIN_ON}",
        model      = model_1d,
        DatasetCls = DopplerDataset1D,
        ckpt_path  = CKPT_1D,
    )
else:
    print("TRAIN_1D=False — skipping 1D training, instantiating model only.")
    model_1d = DopplerTransformer1D().to(DEVICE)

print("Section 2A complete.")


[1D-Attn-v2] Resumed from epoch 11, batch 0  (loss acc = 0.0000, best combined = inf)


/tmp/ipykernel_4141/3633762051.py:23: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[1D-Attn-v2] Ep 011/300:   0%|          | 0/50 [00:00<?, ?batch/s]

### Section 3A — 1D results

In [ ]:
if EVALUATE_1D:
    model_1d = reload_model(DopplerTransformer1D().to(DEVICE), CKPT_1D)

    rows_1d = run_inference(MODEL_NAME_1D, model_1d, DopplerDataset1D)
    df_1d   = pd.DataFrame(rows_1d)
    df_1d.to_csv(CSV_1D, index=False)
    print(f"1D results saved → {CSV_1D}  ({len(df_1d)} rows)")

    print(f"\nAGGREGATE TEST METRICS — {MODEL_NAME_1D}")
    sub      = df_1d[df_1d["model"] == MODEL_NAME_1D]
    path_acc = 100 * sub["path_correct"].mean()
    s_mae    = sub["speed_err"].mean()
    s_med    = sub["speed_err"].median()
    s_bias   = sub["speed_err_signed"].mean()
    d_mae    = sub["dist_err"].mean()
    d_med    = sub["dist_err"].median()
    d_bias   = sub["dist_err_signed"].mean()
    ss_res_s = ((sub["speed_pred"] - sub["speed_gt"]) ** 2).sum()
    ss_tot_s = ((sub["speed_gt"]   - sub["speed_gt"].mean()) ** 2).sum()
    ss_res_d = ((sub["dist_pred"]  - sub["dist_gt"]) ** 2).sum()
    ss_tot_d = ((sub["dist_gt"]    - sub["dist_gt"].mean()) ** 2).sum()
    r2_speed = 1 - ss_res_s / (ss_tot_s + 1e-8)
    r2_dist  = 1 - ss_res_d / (ss_tot_d + 1e-8)

    print(f"\n{MODEL_NAME_1D}  ({len(sub)} test samples)")
    print(f"  Path accuracy     : {path_acc:.2f}%")
    print(f"  Speed MAE         : {s_mae:.3f} m/s  (bias {s_bias:+.3f},  R² {r2_speed:.3f})")
    print(f"  Speed median err  : {s_med:.3f} m/s")
    print(f"  Distance MAE      : {d_mae:.3f} m   (bias {d_bias:+.3f},  R² {r2_dist:.3f})")
    print(f"  Distance median   : {d_med:.3f} m")
    print(f"  Per-path accuracy:")
    for pname in ["straight", "parabola", "bezier"]:
        sub_p = sub[sub["path_gt"] == pname]
        if len(sub_p):
            print(f"    {pname:10s}: {100*sub_p['path_correct'].mean():.1f}%  (n={len(sub_p)})")

    summary_1d = pd.DataFrame([{
        "Model": MODEL_NAME_1D, "Path acc (%)": round(path_acc, 2),
        "Speed MAE (m/s)": round(s_mae, 3), "Speed median": round(s_med, 3),
        "Speed bias": round(s_bias, 3), "Speed R²": round(r2_speed, 4),
        "Dist MAE (m)": round(d_mae, 3), "Dist median": round(d_med, 3),
        "Dist bias": round(d_bias, 3), "Dist R²": round(r2_dist, 4),
    }])
    summary_1d_csv = os.path.join(RESULTS_DIR_1D, "attn_summary_1d.csv")
    summary_1d.to_csv(summary_1d_csv, index=False)
    print(f"\nSummary saved → {summary_1d_csv}")
    print(summary_1d.to_string(index=False))

    plot_results(df_1d, [MODEL_NAME_1D], COLORS, FIG_DIR_1D,
                 SPEED_LIM, DIST_LIM, DIST_BINS, BIN_LABELS)

    print(f"\nEXPORT SUMMARY — 1D")
    print(f"CSV       : {CSV_1D}")
    print(f"Summary   : {summary_1d_csv}")
    print(f"Figures   : {FIG_DIR_1D}")
    print(f"Checkpoint: {CKPT_1D}")
else:
    print("EVALUATE_1D=False — skipping.")


## DopplerCNNTransformer2D

### SECTION 2B — DopplerCNNTransformer2D

In [ ]:
# Architecture:
#   Input       : (B, 1, 84, 432)  — log-CQT spectrogram
#   CNN backbone: 5 Conv2d blocks + 2 MaxPool2d(2×2)
#                 → (B, 128, 21, 108)
#   Freq compress: learned Conv2d(128→64, 21×1) → (B, 64, 1, 108)
#                  squeeze → (B, 64, 108)
#   Project     : Conv1d(64→128, k=1)  → (B, 128, 108)
#   Permute     : (B, 108, 128)
#   + Sinusoidal positional encoding
#   Encoder     : 2-layer Transformer encoder
#                 (nhead=4, dim_feedforward=256, dropout=0.1, norm_first=True)
#   Pool        : mean over time  → (B, 128)
#   Shared      : LayerNorm → Linear(256) → ReLU → Dropout(0.2) → Linear(128) → ReLU
#   Heads       : path (3), speed (1), dist (1)

class DopplerCNNTransformer2D(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2,
                 dim_ff=256, dropout=0.1):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,   32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32,  64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        )
        # After 2× MaxPool2d(2): freq 84→21, time 432→108
        self.freq_compress = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=(21, 1), padding=0),
            nn.BatchNorm2d(64), nn.ReLU(),
        )
        self.proj = nn.Sequential(
            nn.Conv1d(64, d_model, 1), nn.BatchNorm1d(d_model), nn.ReLU(),
        )
        # T' = 108 after pooling
        self.pos_enc = PositionalEncoding(d_model, max_len=200, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.shared = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, d_model), nn.ReLU(),
        )
        self.path_head  = _task_head(d_model, 3)
        self.speed_head = _task_head(d_model, 1)
        self.dist_head  = _task_head(d_model, 1)

    def forward(self, x):
        h = self.cnn(x)               # (B, 128, 21, T')
        h = self.freq_compress(h)     # (B, 64,  1,  T')
        h = h.squeeze(2)              # (B, 64,  T')
        h = self.proj(h)              # (B, 128, T')
        h = h.permute(0, 2, 1)        # (B, T',  128)
        h = self.pos_enc(h)           # (B, T',  128)
        h = self.encoder(h)           # (B, T',  128)
        g = h.mean(dim=1)             # (B, 128)
        z = self.shared(g)
        return (self.path_head(z),
                self.speed_head(z).squeeze(1),
                self.dist_head(z).squeeze(1))


if TRAIN_2D:
    model_2d = DopplerCNNTransformer2D().to(DEVICE)
    model_2d = train_model(
        model_name = f"2D-Attn-{TRAIN_ON}",
        model      = model_2d,
        DatasetCls = DopplerDataset2D,
        ckpt_path  = CKPT_2D,
    )
else:
    print("TRAIN_2D=False — skipping 2D training, instantiating model only.")
    model_2d = DopplerCNNTransformer2D().to(DEVICE)

print("Section 2B complete.")

### Section 3B — 2D results

In [ ]:
if EVALUATE_2D:
    model_2d = reload_model(DopplerCNNTransformer2D().to(DEVICE), CKPT_2D)

    rows_2d = run_inference(MODEL_NAME_2D, model_2d, DopplerDataset2D)
    df_2d   = pd.DataFrame(rows_2d)
    df_2d.to_csv(CSV_2D, index=False)
    print(f"2D results saved → {CSV_2D}  ({len(df_2d)} rows)")

    print(f"\nAGGREGATE TEST METRICS — {MODEL_NAME_2D}")
    sub      = df_2d[df_2d["model"] == MODEL_NAME_2D]
    path_acc = 100 * sub["path_correct"].mean()
    s_mae    = sub["speed_err"].mean()
    s_med    = sub["speed_err"].median()
    s_bias   = sub["speed_err_signed"].mean()
    d_mae    = sub["dist_err"].mean()
    d_med    = sub["dist_err"].median()
    d_bias   = sub["dist_err_signed"].mean()
    ss_res_s = ((sub["speed_pred"] - sub["speed_gt"]) ** 2).sum()
    ss_tot_s = ((sub["speed_gt"]   - sub["speed_gt"].mean()) ** 2).sum()
    ss_res_d = ((sub["dist_pred"]  - sub["dist_gt"]) ** 2).sum()
    ss_tot_d = ((sub["dist_gt"]    - sub["dist_gt"].mean()) ** 2).sum()
    r2_speed = 1 - ss_res_s / (ss_tot_s + 1e-8)
    r2_dist  = 1 - ss_res_d / (ss_tot_d + 1e-8)

    print(f"\n{MODEL_NAME_2D}  ({len(sub)} test samples)")
    print(f"  Path accuracy     : {path_acc:.2f}%")
    print(f"  Speed MAE         : {s_mae:.3f} m/s  (bias {s_bias:+.3f},  R² {r2_speed:.3f})")
    print(f"  Speed median err  : {s_med:.3f} m/s")
    print(f"  Distance MAE      : {d_mae:.3f} m   (bias {d_bias:+.3f},  R² {r2_dist:.3f})")
    print(f"  Distance median   : {d_med:.3f} m")
    print(f"  Per-path accuracy:")
    for pname in ["straight", "parabola", "bezier"]:
        sub_p = sub[sub["path_gt"] == pname]
        if len(sub_p):
            print(f"    {pname:10s}: {100*sub_p['path_correct'].mean():.1f}%  (n={len(sub_p)})")

    summary_2d = pd.DataFrame([{
        "Model": MODEL_NAME_2D, "Path acc (%)": round(path_acc, 2),
        "Speed MAE (m/s)": round(s_mae, 3), "Speed median": round(s_med, 3),
        "Speed bias": round(s_bias, 3), "Speed R²": round(r2_speed, 4),
        "Dist MAE (m)": round(d_mae, 3), "Dist median": round(d_med, 3),
        "Dist bias": round(d_bias, 3), "Dist R²": round(r2_dist, 4),
    }])
    summary_2d_csv = os.path.join(RESULTS_DIR_2D, "attn_summary_2d.csv")
    summary_2d.to_csv(summary_2d_csv, index=False)
    print(f"\nSummary saved → {summary_2d_csv}")
    print(summary_2d.to_string(index=False))

    plot_results(df_2d, [MODEL_NAME_2D], COLORS, FIG_DIR_2D,
                 SPEED_LIM, DIST_LIM, DIST_BINS, BIN_LABELS)

    print(f"\nEXPORT SUMMARY — 2D")
    print(f"CSV       : {CSV_2D}")
    print(f"Summary   : {summary_2d_csv}")
    print(f"Figures   : {FIG_DIR_2D}")
    print(f"Checkpoint: {CKPT_2D}")
else:
    print("EVALUATE_2D=False — skipping.")

## Comparative analysis

In [ ]:
assert os.path.exists(CSV_1D), f"1D results not found at {CSV_1D}"
assert os.path.exists(CSV_2D), f"2D results not found at {CSV_2D}"

df_1d = pd.read_csv(CSV_1D)
df_2d = pd.read_csv(CSV_2D)
df    = pd.concat([df_1d, df_2d], ignore_index=True)

MODEL_NAMES = [MODEL_NAME_1D, MODEL_NAME_2D]
FIG_DIR_CMP = os.path.join(RESULTS_DIR_CMP, "figures")
os.makedirs(FIG_DIR_CMP, exist_ok=True)

# Summary table with R²
summary_rows = []
for mname in MODEL_NAMES:
    sub      = df[df["model"] == mname]
    ss_res_s = ((sub["speed_pred"] - sub["speed_gt"]) ** 2).sum()
    ss_tot_s = ((sub["speed_gt"]   - sub["speed_gt"].mean()) ** 2).sum()
    ss_res_d = ((sub["dist_pred"]  - sub["dist_gt"]) ** 2).sum()
    ss_tot_d = ((sub["dist_gt"]    - sub["dist_gt"].mean()) ** 2).sum()
    summary_rows.append({
        "Model":           mname,
        "Path acc (%)":    round(100 * sub["path_correct"].mean(), 2),
        "Speed MAE (m/s)": round(sub["speed_err"].mean(), 3),
        "Speed median":    round(sub["speed_err"].median(), 3),
        "Speed bias":      round(sub["speed_err_signed"].mean(), 3),
        "Speed R²":        round(1 - ss_res_s / (ss_tot_s + 1e-8), 4),
        "Dist MAE (m)":    round(sub["dist_err"].mean(), 3),
        "Dist median":     round(sub["dist_err"].median(), 3),
        "Dist bias":       round(sub["dist_err_signed"].mean(), 3),
        "Dist R²":         round(1 - ss_res_d / (ss_tot_d + 1e-8), 4),
    })

summary_df  = pd.DataFrame(summary_rows).set_index("Model")
summary_csv = os.path.join(RESULTS_DIR_CMP, "attn_summary_comparison.csv")
summary_df.to_csv(summary_csv)

print("=" * 65)
print(f"COMPARATIVE SUMMARY — {MODEL_NAME_1D} vs {MODEL_NAME_2D}")
print("=" * 65)
print(summary_df.to_string())
print()
for col in summary_df.columns:
    better_is = "max" if col in ("Path acc (%)", "Speed R²", "Dist R²") else "min"
    winner = summary_df[col].idxmax() if better_is == "max" else summary_df[col].idxmin()
    delta  = abs(summary_df[col].diff().dropna().iloc[0])
    print(f"  {col:22s}  winner: {winner}  (gap: {delta:.3f})")
print()

# ── Comparative plots (same layout as CNN benchmark) ─────────────────────────

metrics = [
    ("Path acc (%)",    "Path Accuracy (%)",     True),
    ("Speed MAE (m/s)", "Speed MAE (m/s)",        False),
    ("Speed median",    "Speed Median (m/s)",     False),
    ("Dist MAE (m)",    "Distance MAE (m)",       False),
    ("Dist median",     "Dist Median (m)",        False),
    ("Speed R²",        "Speed R²",               True),
    ("Dist R²",         "Dist R²",                True),
]

fig, axes = plt.subplots(1, len(metrics), figsize=(24, 4.5))
fig.suptitle(f"{MODEL_NAME_1D} vs {MODEL_NAME_2D} — All Metrics", fontsize=13)
x = np.arange(len(MODEL_NAMES))
for ax, (col, label, higher_better) in zip(axes, metrics):
    vals = [summary_df.loc[m, col] for m in MODEL_NAMES]
    bars = ax.bar(x, vals, color=[COLORS[m] for m in MODEL_NAMES],
                  edgecolor="white", linewidth=0.8, width=0.5)
    best = max(vals) if higher_better else min(vals)
    for bar, val, mname in zip(bars, vals, MODEL_NAMES):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01 * max(abs(v) for v in vals),
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)
        if val == best:
            bar.set_edgecolor("gold"); bar.set_linewidth(2.5)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(f"-{TRAIN_ON}", "") for m in MODEL_NAMES], fontsize=8)
    ax.set_ylabel(label, fontsize=8); ax.set_title(label, fontsize=8)
    ax.set_ylim(min(0, min(vals)) - 0.05, max(vals) * 1.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR_CMP, "summary_bars.png"), dpi=150)
plt.show()

# Overlaid scatter
for (val_col, gt_col, lim, fname, title) in [
    ("speed_pred", "speed_gt", SPEED_LIM, "speed_scatter_overlay.png", "Speed Estimation"),
    ("dist_pred",  "dist_gt",  DIST_LIM,  "dist_scatter_overlay.png",  "Distance Estimation"),
]:
    fig, ax = plt.subplots(figsize=(7, 6))
    for mname in MODEL_NAMES:
        sub = df[df["model"] == mname]
        ax.scatter(sub[gt_col], sub[val_col], s=8, alpha=0.35,
                   color=COLORS[mname], label=mname)
    ax.plot(lim, lim, "k--", linewidth=1, label="y = x")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f"GT {gt_col.replace('_gt','')}"); ax.set_ylabel(f"Pred")
    ax.set_title(f"{title} — overlaid"); ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR_CMP, fname), dpi=150)
    plt.show()

# Per-path accuracy
fig, ax = plt.subplots(figsize=(8, 4.5))
path_names = ["straight", "parabola", "bezier"]
x = np.arange(len(path_names)); w = 0.35
for i, mname in enumerate(MODEL_NAMES):
    sub  = df[df["model"] == mname]
    vals = [100 * sub[sub["path_gt"] == p]["path_correct"].mean() for p in path_names]
    bars = ax.bar(x + (i - 0.5) * w, vals, width=w, label=mname,
                  color=COLORS[mname], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(path_names)
ax.set_ylabel("Accuracy (%)"); ax.set_title("Per-Path Classification Accuracy")
ax.set_ylim(0, 115); ax.axhline(100, color="gray", linewidth=0.8, linestyle="--")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR_CMP, "per_path_accuracy.png"), dpi=150)
plt.show()

# Distance MAE by range bin
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(BIN_LABELS)); w = 0.35
for i, mname in enumerate(MODEL_NAMES):
    sub  = df[df["model"] == mname]
    vals = []
    for lo, hi in DIST_BINS:
        mask = (sub["dist_gt"] >= lo) & (sub["dist_gt"] < hi)
        vals.append(sub[mask]["dist_err"].mean() if mask.sum() > 0 else 0.0)
    bars = ax.bar(x + (i - 0.5) * w, vals, width=w, label=mname,
                  color=COLORS[mname], alpha=0.85)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    f"{val:.1f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(BIN_LABELS)
ax.set_xlabel("GT distance range (m)"); ax.set_ylabel("MAE (m)")
ax.set_title("Distance MAE by Range Bin"); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR_CMP, "dist_mae_bins_comparison.png"), dpi=150)
plt.show()

print("\nCOMPARISON EXPORT SUMMARY")
print(f"Summary CSV : {summary_csv}")
print(f"Figures     : {FIG_DIR_CMP}")